# 007 Non-Self Retrieval Evaluation Fixture

This notebook audits the non-self retrieval fixture. It does not run model scoring, score holdout, train, or generate answers.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'data' / 'eval' / 'nonself_retrieval_eval_fixture'
SUMMARY = OUT / 'retrieval_fixture_summary.json'
SUMMARY_MD = OUT / 'retrieval_fixture_summary.md'
QUERY_DOMAINS = OUT / 'retrieval_query_domain_counts.csv'
QUERY_SPLITS = OUT / 'retrieval_query_split_counts.csv'
QREL_GRADES = OUT / 'retrieval_qrel_grade_counts.csv'
QREL_DOMAINS = OUT / 'retrieval_qrel_domain_counts.csv'
PEER_GROUPS = OUT / 'retrieval_peer_group_counts.csv'
POSITIVE_COUNTS = OUT / 'retrieval_positive_counts_by_query.csv'
QRELS = OUT / 'retrieval_qrels.jsonl'

assert SUMMARY.exists(), f'Missing fixture summary: {SUMMARY}'

Leave `RUN_FIXTURE` false unless you are intentionally refreshing the derived fixture artifacts.

In [ ]:
RUN_FIXTURE = False

if RUN_FIXTURE:
    subprocess.run(
        [
            sys.executable,
            str(ROOT / 'scripts' / 'build_nonself_retrieval_eval_fixture.py'),
            '--summary',
        ],
        cwd=ROOT,
        check=True,
    )

## Summary

In [ ]:
display(Markdown(SUMMARY_MD.read_text(encoding='utf-8')))
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
summary['fixture_counts'], summary['positive_summary']

## Query Coverage

In [ ]:
query_domains = pd.read_csv(QUERY_DOMAINS)
display(query_domains)

ax = query_domains.sort_values('records').plot.barh(
    x='primary_domain',
    y='records',
    legend=False,
    figsize=(8, 4),
    color='#3f6f8f',
)
ax.set_title('Retrieval queries by primary domain')
ax.set_xlabel('queries')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Splits And Qrel Grades

In [ ]:
query_splits = pd.read_csv(QUERY_SPLITS)
qrel_grades = pd.read_csv(QREL_GRADES)
display(query_splits)
display(qrel_grades)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
query_splits.plot.bar(x='split', y='records', ax=axes[0], legend=False, color='#6c8c3f')
qrel_grades.plot.bar(x='grade', y='records', ax=axes[1], legend=False, color='#9a5a44')
axes[0].set_title('Query split counts')
axes[0].set_xlabel('')
axes[0].set_ylabel('queries')
axes[1].set_title('Non-self qrel grades')
axes[1].set_xlabel('grade')
axes[1].set_ylabel('qrels')
plt.tight_layout()
plt.show()

## Peer Groups

In [ ]:
peer_groups = pd.read_csv(PEER_GROUPS)
display(peer_groups.head(15))

top_peer_groups = peer_groups.head(15).sort_values('records')
ax = top_peer_groups.plot.barh(
    x='peer_group',
    y='records',
    legend=False,
    figsize=(9, 6),
    color='#3d7b67',
)
ax.set_title('Largest source-query peer groups')
ax.set_xlabel('records')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Positives Per Query

In [ ]:
positive_counts = pd.read_csv(POSITIVE_COUNTS)
display(positive_counts[['positive_records', 'grade_2_positive_records']].describe())

ax = positive_counts['positive_records'].plot.hist(
    bins=24,
    figsize=(8, 4),
    color='#705d9b',
)
ax.set_title('Non-self positives per query')
ax.set_xlabel('positive records')
plt.tight_layout()
plt.show()

## Qrel Samples

In [ ]:
qrels = pd.read_json(QRELS, lines=True)
sample_cols = [
    'query_id',
    'relevant_record_id',
    'relevance_grade',
    'query_primary_domain',
    'match_reasons',
    'shared_source_query_tags',
    'shared_specific_question_tags',
]
display(qrels[qrels['relevance_grade'] == 2][sample_cols].head(10))
display(qrels[qrels['relevance_grade'] == 1][sample_cols].head(10))

## Exit Criteria

- Every retrieval query has at least one non-self positive.
- Query rows are question-only.
- Corpus rows remain retrieval-only and answer generation remains disabled.
- Holdout rows exist for later, but this notebook does not score holdout.
- Next step: score dev queries with flat BGE and triage-filtered BGE against these qrels.